In [ ]:
%load_ext autoreload
%autoreload 2

# Edits to running_axis/*.py take effect on the next cell without restarting the kernel.
# RunningAxis.load() re-reads the axis, so after changing stream.py or axis.py reload the
# object too: axis = RunningAxis.load()


# Running axis — exploration

`running_axis` holds the 1626–1630 archival paragraph stream as **one stateful object**, with any
time span available as a *view* of that one state. It exists because the sequence was being
re-derived per script from dispersed sources, which let three different gap counts circulate for a
single ledger and four correct date counts read as a contradiction.

Two rules the model enforces, both visible below:

1. **A date is an attribute of a session span, not the partition key.** `s6b_known_point_ledger`
   groups by each paragraph's own `date` field and skips any date with none, filing them under a
   fabricated inventory named `no_axis`. That discards 535 dates / 5,590 resolutions corpus-wide —
   **66 dates / 983 resolutions of which already carry a concordance-resolved session.**
2. **No bare integers.** Every figure is a `Tally` naming its population and window, and
   incompatible arithmetic raises rather than quietly producing a number.


## 1. Load once


In [ ]:
import sys, json, collections
from pathlib import Path
from IPython.display import display, HTML

# Ensure repo root is on python path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('repo root:', ROOT)


In [ ]:
from running_axis.axis import RunningAxis
from running_axis.chain import GapLedger, build_brackets
from running_axis.serialize import ribbon_segments, render_html, render_svg, to_json
from running_axis.tally import Window, total, check_partition

axis = RunningAxis.load()          # ~1s: coordinate and claims, text stays on disk
print('coverage:', axis.coverage)
print(f'{len(axis.sessions):,} sessions, {len(axis.dates):,} enriched dates  (coverage-wide)')


## 2. A window's state

`view(start, end)` is the only way to get numbers out, and every line names the window it
describes. October 1628 is the case that prompted the model: the ledger reports it as
*"15 days with an axis, 16 without"*.


In [ ]:
view = axis.view('1628-10-01', '1628-10-31')
for name, tally in view.report().items():
    print(tally)


The 16 'without' decompose into three very different things. Only `no_session` is genuinely
absent; `dropped_by_label_grouping` is **already pinned** and lost to the grouping rule alone.


In [ ]:
# the partition is asserted inside report(), not merely claimed here
report = view.report()
check_partition(report['enriched_dates'],
                [report[f'dates_{k}'] for k in
                 ('labelled','dropped_by_label_grouping','nihil_actum','no_session')])
print('partition holds')

for state in view.dates:
    if state.addressability == 'dropped_by_label_grouping':
        s = axis.sessions[state.resolved_session_id]
        print(f'{state.date}  k_e={state.k_e:>3}  ->  {s.session_id}  '
              f'(axis label {",".join(s.label_dates)}, {s.char_length:,} chars)')


## 3. Serialised visualisation

The ribbon is rendered from the **serialised** segments, not from live objects — the same
arrangement as REPUBLIC's `MatchedText.to_dict()` feeding `visualize_as_html`. One strip per
inventory on that inventory's character coordinate; fill is the most consequential claim on each
session, a green border marks a session claimed by more than one date, and stretches no session
covers are left visibly empty. A picture cannot show what the serialised state does not contain,
so an invisible gap means the state really has none.


In [ ]:
display(HTML(render_html(view, title='October 1628 — inventory 3187')))


Hover a block for its session id, character length, axis label and every date claiming it.
The three orange blocks are the days the ledger throws away; two carry a green border because
they host a labelled date *as well*.


In [ ]:
segments = ribbon_segments(view)
print('kinds :', dict(collections.Counter(s.kind for s in segments)))
print('shared:', sum(1 for s in segments if s.shared))
print()
print(to_json(segments[:2], window=str(view.window)))   # the payload the renderer consumes


### A wider window

Scaling up shows how the classes distribute across inventories.


In [ ]:
display(HTML(render_html(axis.view('1628-01-01', '1628-12-31'),
                         title='1628 — all inventories in range')))


## 4. Counts cannot be confused

The four date counts that read as a contradiction are each correct over a different population.


In [ ]:
from running_axis.tally import (AXIS_DATE_LABELS, ENRICHED_DATES,
                                 ENRICHED_WITH_AXIS_LABEL, ENRICHED_WITHOUT_AXIS_LABEL,
                                 AXIS_LABEL_NOT_ENRICHED, Tally, PopulationMismatch)
W = axis.coverage
labels = set()
for stream in axis.streams.values():
    labels |= set(stream.date_labels())
enriched = set(axis.dates)

for population, value in [(AXIS_DATE_LABELS, len(labels)),
                          (ENRICHED_DATES, len(enriched)),
                          (ENRICHED_WITH_AXIS_LABEL, len(labels & enriched)),
                          (ENRICHED_WITHOUT_AXIS_LABEL, len(enriched - labels)),
                          (AXIS_LABEL_NOT_ENRICHED, len(labels - enriched))]:
    print(Tally(population, W, value))


In [ ]:
# the subtraction that was never meaningful
try:
    Tally(AXIS_DATE_LABELS, W, 1240) - Tally(ENRICHED_WITH_AXIS_LABEL, W, 1059)
except PopulationMismatch as exc:
    print('refused:', exc)

# the one that is: a declared partition yields its complement by name
print(Tally(ENRICHED_DATES, W, 1594) - Tally(ENRICHED_WITH_AXIS_LABEL, W, 1059))


In [ ]:
# views compose over disjoint windows, and refuse to over overlapping ones
months = [axis.view(f'1628-{m:02d}-01', f'1628-{m:02d}-31').report()['enriched_resolutions']
          for m in range(1, 13)]
year = axis.view('1628-01-01', '1628-12-31').report()['enriched_resolutions']
print('summed months:', total(months).value, '| year view:', year.value)
try:
    axis.view('1628-01-01','1628-06-30').report()['enriched_dates'] + \
    axis.view('1628-06-01','1628-12-31').report()['enriched_dates']
except PopulationMismatch as exc:
    print('refused:', exc)


## 5. Gaps and brackets

Two structures that are easy to conflate: **gaps** are resolution-level within a day
(`s6b_known_point_ledger`), **brackets** are session-level within an inventory
(`session_offset_eval`). Both are reproduced digit for digit, so they can be trusted where they
disagree with those scripts.


In [ ]:
lines = [json.loads(l) for l in (ROOT / 'output' / 's6b_known_point_ledger.jsonl').read_text().splitlines()]
ledger_report, days = lines[0], [r for r in lines[1:] if r.get('date')]
w = ledger_report['window']
ledger = GapLedger(Window.of(w['start'], w['end']))
for day in days:
    ledger.add_day(day['chain'])

print('window on disk:', w['start'], '..', w['end'])
for name, tally in ledger.report().items():
    print('  ', tally)
print('  shares:', ledger.shares())
print()
print('ledger says:', {k: ledger_report['overall'][k] for k in ('gaps','settled_share_by_gap','open_share_by_resolution')})


Brackets carry a strategic answer: monotonicity alone forces almost nothing, so **shifting**
anchors at session level has very little room before it starts.


In [ ]:
brackets = build_brackets(axis)
print(collections.Counter(b.bracket_class for b in brackets))
print()
print(f'unresolved sittings {sum(b.n_e for b in brackets)} (k_e {sum(b.unresolved_k_e for b in brackets)})')
print('biggest deficits, worst first:')
for b in sorted((b for b in brackets if b.bracket_class.startswith('deficit')),
                key=lambda b: -b.unresolved_k_e)[:10]:
    print(f'  inv {b.inventory}  {b.left_pin_date} -> {b.right_pin_date}  '
          f'n_e={b.n_e} n_h={b.n_h} k_e={b.unresolved_k_e}')


### Where the unplaced mass sits — and a trap to avoid first

The deficit list above is dominated by an inventory **4861** bracket with 191 sittings and
`n_h=0`. It is tempting to conclude the unplaced mass is the secret series. **That conclusion is
false, and the model now makes it hard to state.**

`DateState.candidate_inventory` comes from the concordance's `inventory_id`, which descends from
`session_date_status_1626_1630` — a table holding **3 candidate inventories per date** (4,916 rows
for 1,594 dates). `s6b_known_point_ledger.inventory_of` warns about precisely this. The candidates
for an unplaced date are always *its year's main inventory plus 4562*, because the main series is
annual (3185=1626 … 3189=1630) while 4562 spans the whole period — and the surviving pick tends to
be the secret one. Grouping unplaced dates by it therefore 'discovers' that dates without HTR
belong to the inventories without HTR.

So `DateState.inventory` is **None unless a resolved session establishes it**, and a bracket resting
on candidate attribution says so.


In [ ]:
# the trap, made visible
on_candidate = [b for b in brackets if b.rests_on_candidate_inventory]
print(f'brackets resting on candidate attribution: {len(on_candidate)} of {len(brackets)}')
print(f'  unresolved k_e inside them: {sum(b.unresolved_k_e for b in on_candidate):,}')
print()
unplaced = [s for s in axis.dates.values() if s.addressability == 'no_session']
print(f'no_session dates {len(unplaced)}, resolutions {sum(s.k_e for s in unplaced):,}')
print('how many have an authoritative inventory?',
      sum(1 for s in unplaced if s.inventory is not None),
      '(none can: an unplaced date has no session to read it off)')
print()
print('inventory date spans, inventory read off flat_id (authoritative):')
for inv, stream in sorted(axis.streams.items()):
    labels = stream.date_labels()
    print(f'   {inv}: {labels[0]} .. {labels[-1]}  ({len(labels)} dates)')


The sound route is the XML-derived secret flags (`enriched_resolution_secret_flags`), not the
inventory field. Measured: of the 323 XML-flagged secret dates, **264 are `labelled`** — placed,
with HTR — so being secret does not imply missing HTR. And setting the secret dates aside removes
only **634 of 4,482** resolutions, leaving **298 dates / 3,848 resolutions** that are not secret and
still have no session.

Queried against the ES `full_resolutions` index by `metadata.session_date`, those 298 split:

| | dates | resolutions |
|---|---|---|
| ES has resolutions, our snapshot does not — **recoverable** | 26 | 313 |
| ES has none either — genuinely not transcribed | 272 | 3,535 |

So of the original 4,482: 634 secret (14%), 313 recoverable (7%), 3,535 absent everywhere (79%).


## 6. Mutation keeps the stream in order

`place_anchor` / `shift_anchor` are steps 3a and 3b of the working strategy. They refuse to
break archive order, and a failed shift rolls back rather than leaving the model half-moved.


In [ ]:
from running_axis.axis import MonotonicityError

# session-3185-num-56 carries a 28 March heading inside a 27 March session (archive-verified).
# The ledger gives it one date; the model can hold both.
axis.place_anchor('1626-03-28', 'session-3185-num-56')
shared = axis.sessions['session-3185-num-56']
print(shared.session_id, shared.resolved_dates, 'shared =', shared.is_shared)

try:
    axis.place_anchor('1626-03-28', 'session-3185-num-5')
except MonotonicityError as exc:
    print('refused:', exc)


---

### What this does not do yet

- `place_anchor` takes a **session**, not a character offset inside a paragraph. `spans.py`
  supports sub-paragraph spans, but Group B/C/D anchor supply is not wired to them — which is what
  Track P would need.
- No per-gap remedy classification (additional-anchor-available / shiftable / no-remedy).
- The gap ledger on disk covers one month. A corpus-wide worklist ranked by resolution mass needs
  `uv run python -m scripts.s6b_known_point_ledger` on the full window first.
